In [98]:
import os
import numpy as np
import pandas as pd
from scipy import stats

### Configuration Class

<small>The `Config` class is a simple configuration file used to store parameters for processing the dataset. These parameters are critical for controlling how the you interact with the data. Below is an explanation of the variables defined within this class:

- **RANDOM_SEED**: This is the seed used for random number generation, ensuring reproducibility of results.

- **N_TIME_STEPS**: This refers to the number of time steps (125) in each sequence. The dataset is structured such that every 125 rows correspond to one time step.

- **N_FEATURES**: The number of features in each time step. Here, `2` features are defined — these may represent data columns like `rawData` (rawData), `hr` (heart rate).

- **N_CLASSES**: This defines the number of classes the dataset will be categorized into. Here, `3` classes represent the different phases of the seizure detection process (e.g., Normal, Pre-Ictal, Ictal).

This class helps maintain the structure of the model and ensures that key parameters are easy to adjust for experimentation.

</small>


In [99]:
class Config:

    RANDOM_SEED = 333
    N_TIME_STEPS = 125   # 125 datapoints per timestep
    N_FEATURES = 2      # rawData,ppg
    step = 100           # window overlap = 50 -10 = 40  (80% overlap)
    N_CLASSES = 3       # Normal, Pre-Ictal and Ictal

### DataLoader Class

<small>The `DataLoader` class is responsible for loading and preparing the dataset for training or evaluation. It processes the raw time series data, splits it into segments based on time steps, and assigns labels. Here's a breakdown of the variables and methods in this class:

- **`__init__` Method:**
    - **`dataframe`**: The input dataset, typically a Pandas DataFrame, containing the raw time series data.
    - **`time_steps`**: The length of each data segment in terms of the number of time steps (records) that will be used as input to the model.
    - **`step`**: The step size (overlap) used when slicing the time series into smaller windows.
    - **`target_column`**: The column name in the dataset that contains the class labels (e.g., seizure phases like Normal, Pre-Ictal, Ictal).

- **`load_data` Method:**
    - This method processes the dataset by:
      - **Grouping the data by eventID**: This ensures that the data for each event is kept together and not mixed with other events.
      - **Iterating over each group**: For each event, it checks if the event has enough data (at least `time_steps` number of records).
      - **Slicing the data into smaller segments**: Data is divided into smaller windows of size `time_steps`, with the window sliding by `step` each time.
      - **Extracting features**: For each segment, it extracts the `rawData` and `ppg` (heart rate) values, combining them into a single feature set for the model.
      - **Assigning labels**: For each segment, it uses the mode of the labels in the target column to assign a label to the segment. The mode is used to avoid ambiguity in case the segment has mixed labels.
      - **Storing event and user information**: The event ID and user ID are saved along with the data segments and labels.

- **Data Output**:
    - **`segments`**: The time series data, organized into segments of size `time_steps`.
    - **`labels`**: One-hot encoded labels for each segment, corresponding to the target column.
    - **DataFrame Output**: The processed data is returned as a DataFrame containing the segments, their labels, and the associated event and user IDs.
</small>

In [100]:
class DataLoader:
    def __init__(self, dataframe, time_steps, step, target_column):
        self.dataframe = dataframe
        self.time_steps = time_steps
        self.step = step
        self.target_column = target_column

    def load_data(self):
        segments = []
        labels = []
        event_ids = []
        user_ids = []

        # Group data by eventID to ensure events are kept intact
        grouped = self.dataframe.groupby('eventId')

        for event_id, group in grouped:
            if len(group) >= self.time_steps:  # Process if the event group has enough data
                for i in range(0, len(group) - self.time_steps + 1, self.step):
                    mag = group['rawData'].values[i: i + self.time_steps]
                    hr = group['ppg'].values[i: i + self.time_steps]
                    segment = np.column_stack((mag, hr))  # Combine magnitude and heart rate features
                    label_mode = stats.mode(group[self.target_column][i: i + self.time_steps])
                    if isinstance(label_mode.mode, np.ndarray):
                        label = label_mode.mode[0]
                    else:
                        label = label_mode.mode

                    segments.append(segment)
                    labels.append(label)
                    event_ids.append(event_id)
                    user_ids.append(group['userID'].iloc[0])  # Assuming userID is consistent within an event

        # Convert to numpy arrays
        segments = np.asarray(segments, dtype=np.float32)
        labels = np.asarray(pd.get_dummies(labels), dtype=np.float32)

        # Create DataFrame to store eventID and userID alongside segments and labels
        df_labels = pd.DataFrame({
            'segments': list(segments),
            'labels': list(labels),
            'eventId': event_ids,
            'userID': user_ids
        })

        return df_labels



<small>
### Data Loading and Processing

In this section, we first load the dataset from a CSV file and then use the `DataLoader` class to process it. Below is a breakdown of the key steps:

1. **Reading the Dataset**:
    - **`sample_dataset_path = 'Data/sample_dataset.csv'`**: The file path to the raw dataset stored in the `Data` folder.
    - **`target_column = 'label'`**: The column containing the class labels (e.g., seizure phases).
    - **`df = pd.read_csv(sample_dataset_path)`**: This reads the dataset into a Pandas DataFrame, allowing easy manipulation and inspection.
    - **`df.head()`**: Displays the first few rows of the dataset to verify that it has been loaded correctly.

2. **Using the DataLoader Class**:
    - The `DataLoader` is initialized with:
        - **`dataframe=df`**: The loaded dataset.
        - **`time_steps=Config.N_TIME_STEPS`**: The number of time steps per segment.
        - **`step=Config.step`**: The step size (overlap) for sliding the time window.
        - **`target_column='label'`**: The column containing class labels.
    - **`load_data()`**: Processes the data into smaller segments, assigns labels, and returns a DataFrame (`df_sample`) containing:
        - **`segments`**: Data windows for model input.
        - **`labels`**: One-hot encoded class labels for each segment.
        - **`eventId`**: Event ID corresponding to each segment.
        - **`userID`**: User ID associated with each segment.
    - **`df_sample.head()`**: Displays the first few rows of the processed data to verify its structure.

This process prepares the dataset for training or evaluation by splitting it into time series segments and assigning labels.

</small>

In [101]:
# Reading data from Google Drive
sample_dataset_path = '../Data/sample_dataset.csv'

# Name of the target colum
target_column = 'label'

# Load sample data as csv using Pandas lib
df = pd.read_csv(sample_dataset_path)

#Print the first 5 rows of the dataset
df.head(5)

,Id,eventId,userID,rawData,ppg,type,subType,label
0,1,5635,45,1630.0,69.0000,Seizure,Aura,0
1,2,5635,45,1632.0,68.9283,Seizure,Aura,0
2,3,5635,45,1628.0,68.8575,Seizure,Aura,0
3,4,5635,45,1621.0,68.7877,Seizure,Aura,0
4,5,5635,45,1624.0,68.7189,Seizure,Aura,0


In [102]:
# Initialize DataLoader
data_loader = DataLoader(dataframe=df, time_steps=Config.N_TIME_STEPS, step=Config.step, target_column='label')

# Load data (this will return a DataFrame with segments, labels, eventID, and userID)
df_sample = data_loader.load_data()

#Print DataLoader Output
df_sample.head()

,segments,labels,eventId,userID
0,"[[1066.9883, 16.4196], [1007.7778, 15.768], [1...","[0.0, 1.0, 0.0]",115,39
1,"[[1054.6431, -17.4891], [1058.0813, -17.4321],...","[0.0, 1.0, 0.0]",115,39
2,"[[1057.4006, 31.9105], [1027.9144, 32.7409], [...","[0.0, 1.0, 0.0]",115,39
3,"[[1060.9882, 95.0641], [1057.8015, 95.2513], [...","[0.0, 1.0, 0.0]",115,39
4,"[[1300.1477, 87.5605], [1059.5396, 87.4359], [...","[0.0, 1.0, 0.0]",115,39
